In [5]:

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import seaborn as sns
from matplotlib.image import imread
from PIL import Image
import tensorflow as tf
np.random.seed(1337)
import gc

from tensorflow.keras import layers
from keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Activation, Dropout, AveragePooling2D,Flatten, Dense, Conv2D,MaxPool2D, MaxPooling2D, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
import warnings
warnings.filterwarnings('ignore')


img_size = 128
print(os.listdir())
dataset = os.listdir("music_dataset_spectro_3_instrument")
labels = dataset
print(labels)

['.git', '.vscode', 'model_classes_3.ipynb', 'music_dataset', 'music_dataset_spectro_3_instrument', 'README.md', 'spectrogramMaker.py']
['test', 'train', 'valid']


In [6]:
def get_dataset_array(data_dir):
    data = []
    for label in labels:
        print(label)
        path = os.path.join(data_dir, label)
        class_num = labels.index(label)
        for img in os.listdir(path):
            try:
                img_arr = cv2.imread(os.path.join(path, img))
                resized_arr = cv2.resize(img_arr, (img_size, img_size))
                data.append([resized_arr, class_num])
                gc.collect()
            except Exception as e:
                print(e)
    return np.array(data,dtype="object")

In [7]:
train = get_dataset_array("music_dataset_spectro_3_instrument/train/")
test = get_dataset_array("music_dataset_spectro_3_instrument/test/")
valid = get_dataset_array("music_dataset_spectro_3_instrument/valid/")

test


FileNotFoundError: [WinError 3] The system cannot find the path specified: 'music_dataset_spectro_3_instrument/train/test'

In [ ]:
print(train.shape)
print(test.shape)
print(valid.shape)

(32254, 2)
(3905, 2)
(4020, 2)


In [ ]:
x_train = []
y_train = []

x_val = []
y_val = []

x_test = []
y_test = []

for feature, label in train:
    x_train.append(feature)
    y_train.append(label)

for feature, label in test:
    x_test.append(feature)
    y_test.append(label)

for feature, label in valid:
    x_val.append(feature)
    y_val.append(label)
     
del train
del test
del valid

In [ ]:
gc.collect()
x_train = np.array(x_train)/255
gc.collect()
x_test = np.array(x_test)/255
gc.collect()
x_val = np.array(x_val)/255
gc.collect()

0

In [ ]:
x_train = x_train.reshape(-1, img_size, img_size, 3)
y_train = np.array(y_train)

x_val = x_val.reshape(-1, img_size, img_size, 3)
y_val = np.array(y_val)

x_test = x_test.reshape(-1, img_size, img_size, 3)
y_test = np.array(y_test)


In [ ]:
# Model setup
model = Sequential()
model.add(Conv2D(32, (3,3), activation = 'relu', input_shape = (img_size, img_size, 3)))
model.add(MaxPool2D((2,2)))
model.add(BatchNormalization())
model.add(Conv2D(64, (3,3), activation = 'relu', input_shape = (img_size, img_size, 3)))
model.add(MaxPool2D((2,2)))
model.add(BatchNormalization())
model.add(Conv2D(128, (3,3), activation = 'relu', input_shape = (img_size, img_size, 3)))
model.add(MaxPool2D((2,2)))
model.add(BatchNormalization())

model.add(Flatten())
model.add(Dense(units = 128, activation = 'relu'))
model.add(Dropout(0.5))
model.add(Dense(units = 22, activation = 'softmax'))

model.compile(
              optimizer = 'adam', loss = 'sparse_categorical_crossentropy',
              metrics = ['accuracy']
              )
     

In [ ]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 63, 63, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 30, 30, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 28, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 14, 14, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     3,211,392 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 22)             │         2,838 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,308,374 (12.62 MB)

 Trainable params: 3,307,926 (12.62 MB)

 Non-trainable params: 448 (1.75 KB)

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(

      featurewise_center=False,
      samplewise_center=False,
      featurewise_std_normalization=False,
      samplewise_std_normalization=False,
      zca_whitening=False,
      rotation_range = 30,
      zoom_range = 0.2,

      width_shift_range = 0.1,
      height_shift_range = 0.1,
      horizontal_flip = True,
      vertical_flip=False)


datagen.fit(x_train)

In [ ]:
learning_rate_reduction = ReduceLROnPlateau(monitor = 'val_accuracy', patience = 2, verbose = 1, factor = 0.3, min_lr = 0.000001)

In [ ]:
batch_size = 32
n_epochs = 25
model.fit(datagen.flow(x_train, y_train, batch_size = batch_size),
                    epochs = n_epochs, validation_data = datagen.flow(x_val, y_val),
                    callbacks = [learning_rate_reduction])

Epoch 1/25
1008/1008 ━━━━━━━━━━━━━━━━━━━━ 187s 182ms/step - accuracy: 0.3801 - loss: 2.1640 - val_accuracy: 0.4963 - val_loss: 2.2995 - learning_rate: 0.0010
Epoch 2/25
1008/1008 ━━━━━━━━━━━━━━━━━━━━ 170s 168ms/step - accuracy: 0.4740 - loss: 1.7095 - val_accuracy: 0.4614 - val_loss: 2.1011 - learning_rate: 0.0010
Epoch 3/25
1008/1008 ━━━━━━━━━━━━━━━━━━━━ 0s 161ms/step - accuracy: 0.5027 - loss: 1.5675
Epoch 3: ReduceLROnPlateau reducing learning rate to 0.0003000000142492354.
1008/1008 ━━━━━━━━━━━━━━━━━━━━ 171s 169ms/step - accuracy: 0.5117 - loss: 1.5294 - val_accuracy: 0.4863 - val_loss: 3.2521 - learning_rate: 0.0010
Epoch 4/25
1008/1008 ━━━━━━━━━━━━━━━━━━━━ 169s 168ms/step - accuracy: 0.5646 - loss: 1.3218 - val_accuracy: 0.6667 - val_loss: 0.9603 - learning_rate: 3.0000e-04
Epoch 5/25
1008/1008 ━━━━━━━━━━━━━━━━━━━━ 170s 169ms/step - accuracy: 0.5929 - loss: 1.2409 - val_accuracy: 0.6781 - val_loss: 0.9228 - learning_rate: 3.0000e-04
Epoch 6/25
1008/1008 ━━━━━━━━━━━━━━━━━━━━ 170s 

In [ ]:
gc.collect()
print("Accuracy of the model is - " , model.evaluate(x_test,y_test)[1]*100 , "%")
model.save('my_model_3.keras')

123/123 ━━━━━━━━━━━━━━━━━━━━ 4s 35ms/step - accuracy: 0.9014 - loss: 0.5626
Accuracy of the model is -  90.14084339141846 %
